# Generated sources: compare variants of a generator

A design can have its sources written by a program, its generator. Xeda runs the generator in the
launch, before the first flow. The generated files go to a storage under the run root. Xeda never
writes in the design tree.

This notebook writes a small design with a Python generator. The generator makes a counter of a
given width. The notebook runs three widths in one sweep, compares them, and delivers the source of
one variant into a directory.

The notebook needs Yosys and `nextpnr-ecp5`.

In [ ]:
import os
from pathlib import Path

import xeda

EXAMPLES = Path(os.environ.get("XEDA_EXAMPLES", Path.cwd().parent / "examples")).resolve()
if not EXAMPLES.is_dir():
    raise FileNotFoundError(f"No examples in {EXAMPLES}. Set XEDA_EXAMPLES to the examples directory.")

WORK = Path("notebook-output").resolve()
ws = xeda.Workspace(WORK / "xeda_run", hashed_run_dirs=True)

## Write the design

The generator takes two arguments: the width and the storage. The design file names the generator
and lists the file that it writes in `rtl.sources`. `$XEDA_GENERATED` is the storage.

In [ ]:
import sys

DESIGN_DIR = WORK / "design"
DESIGN_DIR.mkdir(parents=True, exist_ok=True)

GENERATOR = """\
import sys
from pathlib import Path

width, storage = int(sys.argv[1]), Path(sys.argv[2])
storage.mkdir(parents=True, exist_ok=True)
lines = [
    "module counter (",
    "    input wire clk,",
    "    input wire reset,",
    f"    output reg [{width - 1}:0] count",
    ");",
    "  always @(posedge clk)",
    "    if (reset) count <= 0;",
    "    else count <= count + 1;",
    "endmodule",
]
(storage / "counter.v").write_text("\\n".join(lines) + "\\n")
"""

DESIGN = f"""\
name: counter
rtl:
  sources:
    - $XEDA_GENERATED/counter.v
  top: counter
  clock.port: clk
  generator:
    executable: {sys.executable}
    args: [gen_counter.py, "8", $XEDA_GENERATED]
    sources: [gen_counter.py]
flows:
  yosys_fpga:
    fpga.part: LFE5U-85F-6BG381C
    clock.period: 10 ns
  nextpnr:
    lpf_allow_unconstrained: true
"""

(DESIGN_DIR / "gen_counter.py").write_text(GENERATOR)
(DESIGN_DIR / "counter.xeda.yaml").write_text(DESIGN)

## Make the variants

Loading the design runs nothing. A variant is the design with other `args` for the generator.

Each variant has its own design name. Each generator recipe is its own run, with its own run
directory and its own storage.

The variant of width 16 also sets `rtl.generator.outputs_to`. At the end of a launch that passed,
Xeda copies the generated sources into that directory.

In [ ]:
from xeda import Design, Spec

base = Design.load(DESIGN_DIR / "counter.xeda.yaml")
DELIVERED = WORK / "delivered"


def variant(width: int, **updates) -> Spec:
    design = base.with_updates(
        {
            "name": f"counter{width}",
            "rtl.generator.args": ["gen_counter.py", str(width), "$XEDA_GENERATED"],
            **updates,
        }
    )
    return Spec("yosys_fpga+nextpnr", design)


specs = [
    variant(8),
    variant(16, **{"rtl.generator.outputs_to": str(DELIVERED)}),
    variant(32),
]

## Run the sweep

The sweep runs the generator once for each variant, and then the flows. It compares the variants
in one table.

In [ ]:
sweep = ws.sweep(specs)
print({outcome.value: count for outcome, count in sweep.counts().items()})

In [ ]:
sweep.to_pandas(metrics=("nextpnr.lut", "nextpnr.fmax"))[["design", "nextpnr.lut", "nextpnr.fmax"]]

## Read the generated sources

The record of a run lists the generated sources that the run read. `open` gives the files, and
they hold the bytes that the run read. Read them inside the `with` block.

In [ ]:
generated = {}
for p in sweep:
    with p.result.nodes["yosys_fpga"].record.open() as run:
        generated[p.spec.design.name] = run.generated_sources["$XEDA_GENERATED/counter.v"].read_bytes()

for name, data in generated.items():
    print(name, len(data), data.splitlines()[0].decode())

## Check the delivery

The delivered file is a copy of what the variant of width 16 read.

In [ ]:
delivered = (DELIVERED / "counter.v").read_bytes()
print(delivered == generated["counter16"])
print([(d.key, d.delivered) for d in sweep.points[1].result.nodes["yosys_fpga"].deliveries])